# Data Preprocessing

This notebook is made to generate the all proper csv files, necessary for our own visualizations.

More comments on data analysis related to each specific chart and the reasons for our choices are well documented in the notebooks related with each individual chart.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
def load_data(file_path, sep=',', encoding='utf-8'):
    """Load data from a CSV file into a pandas DataFrame."""
    try:
        data = pd.read_csv(file_path, sep=sep, encoding=encoding)
        print("Data loaded successfully.")
        return data
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return None

def summarize_data(data):
    """Generate summary statistics of the DataFrame."""
    if data is not None:
        summary = data.describe()
        print("Data summary:")
        print(summary)
        return summary
    else:
        print("No data to summarize.")
        return None

## Preprocessing the GDELT csv

Since the csv is very large, we first converted it to a parquet file and then split it into smaller parquet files, each containing specific metrics.

The primary key is `conflict_id`, which is used to join these files later in the visualization notebooks.

Another key is the quintuple `['mention_week', 'conflict_country', 'actor1_country', 'actor2_country', 'media_country']`, which is used to aggregate mentions on a weekly basis per conflict and country.

In [3]:
df_mentions = pd.read_parquet('../data/GDELT/conflict_data_final.parquet')
df_mentions.head()

,mention_week,conflict_country,actor1_country,actor2_country,media_country,mentions_count,distinct_events,distinct_media_sources,verbal_conflict_mentions,material_conflict_mentions,...,international_mentions,international_unique_events,avg_tone,stddev_tone,median_tone,avg_impact,stddev_impact,median_impact,top_event_article_url,most_impactful_event_article_url
0,2026-01-12,UK,UNK,GBR,UP,16,13,11,3,13,...,0,0,-2.480499,2.584627,-2.956522,-6.337500,3.286919,-7.2,https://www.unian.ua/world/britaniya-hoche-fin...,https://www.eurointegration.com.ua/news/2026/0...
1,2026-01-12,FR,FRA,USA,US,101,28,71,41,60,...,0,0,-2.311362,3.035549,-0.337079,-5.360396,2.004599,-5.0,https://www.stltoday.com/news/local/crime-cour...,https://www.thenation.com/article/activism/pue...
2,2026-01-12,IS,GBR,UNK,UK,8,8,5,6,2,...,0,0,-5.187022,3.175163,-4.311774,-3.687500,2.463121,-2.0,https://www.cityam.com/londons-record-low-conf...,https://www.thetimes.com/uk/politics/article/b...
3,2026-01-12,UP,EUR,RUS,UK,208,5,105,207,1,...,0,0,-4.748191,0.274187,-4.770318,-2.052885,0.591478,-2.0,https://www.andoveradvertiser.co.uk/news/natio...,https://www.thetimes.com/world/russia-ukraine-...
4,2026-01-12,EZ,SVK,UNK,EZ,6,5,4,5,1,...,0,0,-5.115266,2.634168,-6.008584,-3.750000,2.752272,-2.0,https://www.denik.cz/ekonomika/nejsem-na-briga...,https://www.denik.cz/ekonomika/nejsem-na-briga...


In [23]:
df_mentions.columns

Index(['mention_week', 'conflict_country', 'actor1_country', 'actor2_country',
       'media_country', 'mentions_count', 'distinct_events',
       'distinct_media_sources', 'verbal_conflict_mentions',
       'material_conflict_mentions', 'verbal_conflict_unique_events',
       'material_conflict_unique_events', 'state_mentions',
       'state_unique_events', 'insurgents_mentions',
       'insurgents_unique_events', 'civilians_mentions',
       'civilians_unique_events', 'international_mentions',
       'international_unique_events', 'avg_tone', 'stddev_tone', 'median_tone',
       'avg_impact', 'stddev_impact', 'median_impact', 'top_event_article_url',
       'most_impactful_event_article_url'],
      dtype='object')

In [22]:
len(df_mentions.columns)

28

In [18]:
df_mentions.iloc[3]['top_event_article_url']

'https://www.andoveradvertiser.co.uk/news/national/25760295.us-accuses-russia-dangerous-inexplicable-escalation-war-ukraine/'

In [19]:
df_mentions.iloc[3]['most_impactful_event_article_url']

'https://www.thetimes.com/world/russia-ukraine-war/article/russia-ukraine-invasion-longer-soviet-union-ww2-involvement-g8gb2ljw5'

In [20]:
# show the first row and show all columns!
pd.set_option('display.max_columns', None)
df_mentions.iloc[3]

mention_week                                                               2026-01-12
conflict_country                                                                   UP
actor1_country                                                                    EUR
actor2_country                                                                    RUS
media_country                                                                      UK
mentions_count                                                                    208
distinct_events                                                                     5
distinct_media_sources                                                            105
verbal_conflict_mentions                                                          207
material_conflict_mentions                                                          1
verbal_conflict_unique_events                                                       4
material_conflict_unique_events                       

In [ ]:
# Do not run these cells, they are for context only

# --- IGNORE ---
# path_mentions = "../data/GDELT/conflict_data_final.csv"
# df_mentions = pd.read_csv(path_mentions)
# df_mentions.to_parquet("../data/GDELT/conflict_data_final.parquet", index=False)
# key_cols = ['mention_week', 'conflict_country', 'actor1_country', 'actor2_country', 'media_country']
# total_rows = len(df_mentions)
# unique_combinations = df_mentions[key_cols].drop_duplicates().shape[0]
# df_keys = df_mentions[key_cols].drop_duplicates().reset_index(drop=True)
# df_keys['conflict_id'] = df_keys.index
# df_mentions = df_mentions.merge(df_keys, on=key_cols, how='left')


# Now create normalized tables:

# save_path = "../data/GDELT/"

# 1. Dimension table (the keys + ID)
# df_dimensions = df_mentions[['conflict_id'] + key_cols].drop_duplicates()
# df_dimensions.to_parquet(save_path + "gdelt_keys.parquet", index=False)

# 2. Volume metrics (counts and mentions)
# df_volume = df_mentions[['conflict_id', 'mentions_count', 'distinct_events', 'distinct_media_sources']]
# df_volume.to_parquet(save_path + "gdelt_volume_metrics.parquet", index=False)

# 3. Conflict type metrics (verbal vs material)
# df_conflict_types = df_mentions[['conflict_id', 
#                         'verbal_conflict_mentions', 'material_conflict_mentions',
#                         'verbal_conflict_unique_events', 'material_conflict_unique_events']]
# df_conflict_types.to_parquet(save_path + "gdelt_conflict_types.parquet", index=False)

# 4. Actor type metrics (state, insurgents, civilians, international)
# df_actors = df_mentions[['conflict_id',
#                 'state_mentions', 'state_unique_events',
#                 'insurgents_mentions', 'insurgents_unique_events',
#                 'civilians_mentions', 'civilians_unique_events',
#                 'international_mentions', 'international_unique_events']]
# df_actors.to_parquet(save_path + "gdelt_actor_metrics.parquet", index=False)

# 5. Sentiment/tone metrics
# df_sentiment = df_mentions[['conflict_id', 
#                    'avg_tone', 'stddev_tone', 'median_tone']]
# df_sentiment.to_parquet(save_path + "gdelt_sentiment.parquet", index=False)

# 6. Impact metrics
# df_impact = df_mentions[['conflict_id',
#                 'avg_impact', 'stddev_impact', 'median_impact']]
# df_impact.to_parquet(save_path + "gdelt_impact.parquet", index=False)

# 7. Article references (URLs)
# df_articles = df_mentions[['conflict_id', 
#                   'top_event_article_url', 'most_impactful_event_article_url']]
# df_articles.to_parquet(save_path + "gdelt_articles.parquet", index=False)

# print("Files created:")
# print(f"- gdelt_keys.parquet ({df_dimensions.shape})")
# print(f"- gdelt_volume_metrics.parquet ({df_volume.shape})")
# print(f"- gdelt_conflict_types.parquet ({df_conflict_types.shape})")
# print(f"- gdelt_actor_metrics.parquet ({df_actors.shape})")
# print(f"- gdelt_sentiment.parquet ({df_sentiment.shape})")
# print(f"- gdelt_impact.parquet ({df_impact.shape})")
# print(f"- gdelt_articles.parquet ({df_articles.shape})")

## Waffle Chart

In [6]:
# load data from ../data/GDELT/2021-news-outlets-by-countrycode-2015-2021.csv
data_file_path = '../data/GDELT/2021-news-outlets-by-countrycode-2015-2021.csv'
df = load_data(data_file_path)
df.head()

# Group by 'domain' and select the 'countrycode' column with the max 'cnt' value
df_top_countries = df.loc[df.groupby('domain')['cnt'].idxmax()][['domain', 'countrycode']]
df_top_countries.head()

# Save the processed data to a new CSV file
#output_file_path = '../data/GDELT/top_countries_by_domain.csv'
#df_top_countries.to_csv(output_file_path, index=False)

Data loaded successfully.


,domain,countrycode
0,00221.info,SG
5,007.com,UK
10,007calcio.com,IT
11,007collector.com,AU
13,007spain.com,SN


## Methodology Points

https://blog.gdeltproject.org/mapping-the-media-a-geographic-lookup-of-gdelts-sources-2015-2021/

https://parusanalytics.com/eventdata/cameo.dir/CAMEO.Manual.1.1b3.pdf

https://unstats.un.org/unsd/methodology/m49/

https://github.com/mongodb-developer/gdelt-primer?tab=readme-ov-file

https://arpieb.com/2018/06/20/digging-into-the-gdelt-event-schema/

https://blog.gdeltproject.org/mapping-the-media-a-geographic-lookup-of-gdelts-sources/


## Cleaning the FIPS file and making sure that it is aligned with the ACLED datasets

In [12]:
# Reading the json file with country codes associated to country names in ../src/json/fips.json
with open('../src/json/fips.json', 'r') as f:
    fips = json.load(f)

# Creating a DataFrame from the fips dictionary
fips_df = pd.DataFrame.from_dict(fips, orient='index')
fips_df.reset_index(inplace=True)
fips_df.columns = ['fips_code', 'country_name']
fips_df.head()

,fips_code,country_name
0,AA,Aruba
1,AC,Antigua and Barbuda
2,AD,Akrotiri and Dhekelia
3,AE,United Arab Emirates
4,AF,Afghanistan


In [13]:
# Reading the ACLED datasets
acled_path = '../data/ACLED/number_of_reported_fatalities_by_country-year_as-of-12Dec2025.csv'
acled_df = pd.read_csv(acled_path, encoding='utf-8', sep=';')
acled_df.head()

,COUNTRY,YEAR,FATALITIES
0,Afghanistan,2017,36360
1,Afghanistan,2018,42991
2,Afghanistan,2019,41419
3,Afghanistan,2020,30977
4,Afghanistan,2021,42425


In [14]:
# Now we need to make sure that the country names in acled_df match those in fips_df
acled_countries = set(acled_df['COUNTRY'].unique())
fips_countries = set(fips_df['country_name'].unique())
print("Countries in ACLED dataset not in FIPS dataset:")
print(acled_countries - fips_countries)

# Obtain the missing countries and make a list
missing_countries = list(acled_countries - fips_countries)

Countries in ACLED dataset not in FIPS dataset:
set()


In [6]:
# Find the row in acled_df where COUNTRY is 'Bailiwick of Jersey' then do the same in fips_df
print(acled_df[acled_df['COUNTRY'] == 'Bailiwick of Jersey'])

                 COUNTRY  YEAR  FATALITIES
177  Bailiwick of Jersey  2020           0
178  Bailiwick of Jersey  2021           0
179  Bailiwick of Jersey  2025           0


In [7]:
# Loading the json which generated the fips.json file
# they have already been edited to include the missing countries
# so we can use them to create an updated fips.json file

with open('../src/json/fips_extension.json', 'r') as f:
    fips_extension = json.load(f)

with open('../src/json/fips_to_country.json', 'r') as f:
    fips_to_country = json.load(f)


In [8]:
print(fips_to_country['WZ'])

eSwatini


In [9]:
# Merge the two dictionaries
fips_updated = {**fips_to_country, **fips_extension}
# Save the updated dictionary to a new json file
with open('../src/json/fips.json', 'w') as f:
    json.dump(fips_updated, f, indent=4)

In [11]:
print(fips_updated['WZ'])

eSwatini


In [22]:
# Now load geodata: ../src/json/world.json
with open('../src/json/world.json', 'r') as f:
    world_geo = json.load(f)
print(world_geo['features'][0])

{'type': 'Feature', 'properties': {'name': 'Afghanistan'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[61.210817, 35.650072], [62.230651, 35.270664], [62.984662, 35.404041], [63.193538, 35.857166], [63.982896, 36.007957], [64.546479, 36.312073], [64.746105, 37.111818], [65.588948, 37.305217], [65.745631, 37.661164], [66.217385, 37.39379], [66.518607, 37.362784], [67.075782, 37.356144], [67.83, 37.144994], [68.135562, 37.023115], [68.859446, 37.344336], [69.196273, 37.151144], [69.518785, 37.608997], [70.116578, 37.588223], [70.270574, 37.735165], [70.376304, 38.138396], [70.806821, 38.486282], [71.348131, 38.258905], [71.239404, 37.953265], [71.541918, 37.905774], [71.448693, 37.065645], [71.844638, 36.738171], [72.193041, 36.948288], [72.63689, 37.047558], [73.260056, 37.495257], [73.948696, 37.421566], [74.980002, 37.41999], [75.158028, 37.133031], [74.575893, 37.020841], [74.067552, 36.836176], [72.920025, 36.720007], [71.846292, 36.509942], [71.262348, 36.074388], [71.498768,

In [23]:
# Take the list of 'name' properties from world_geo and compare to acled_countries
geo_countries = set([feature['properties']['name'] for feature in world_geo['features']])
print("Countries in ACLED dataset not in Geo dataset:")
print(acled_countries - geo_countries)

# Make it a list
missing_geo_countries = list(acled_countries - geo_countries)

Countries in ACLED dataset not in Geo dataset:
{'Tokelau', 'Sao Tome and Principe', 'Turks and Caicos Islands', 'Anguilla', 'Saint-Barthelemy', 'Niue', 'Reunion', 'Saint Vincent and the Grenadines', 'Saint-Martin', 'South Georgia and the South Sandwich Islands', 'Saint Pierre and Miquelon', 'Saint Lucia', 'Grenada', 'Wallis and Futuna', 'Cook Islands', 'Guadeloupe', 'Andorra', 'Faroe Islands', 'Mauritius', 'Marshall Islands', 'Comoros', 'Antigua and Barbuda', 'Saint Helena, Ascension and Tristan da Cunha', 'Pitcairn', 'Dominica', 'Barbados', 'Isle of Man', 'Martinique', 'Gibraltar', 'British Virgin Islands', 'Norfolk Island', 'Monaco', 'British Indian Ocean Territory', 'Bahrain', 'United States Minor Outlying Islands', 'Seychelles', 'Cocos (Keeling) Islands', 'Kiribati', 'Bermuda', 'Micronesia', 'Mayotte', 'Montserrat', 'Vatican City', 'Tuvalu', 'Caribbean Netherlands', 'Saint Kitts and Nevis', 'Cape Verde', 'Cayman Islands', 'Akrotiri and Dhekelia', 'French Polynesia', 'Aruba', 'Baili

In [24]:
print(geo_countries - acled_countries)

{'Western Sahara', 'Somaliland', 'West Bank', 'Northern Cyprus'}
